# Beauty Image Embeddings with MobileCLIP2-S0

This notebook generates image embeddings for `Beauty_and_Personal_Care` metadata using:

```text
Model: apple/MobileCLIP2-S0
Device: mps
Storage dtype: float32
CBPR projection later: raw image_embedding_dim -> 32
```

The notebook follows the procedure in `Beauty_image_embedding_generating.md`:

1. Reproduce the Beauty interaction filtering logic from Yun Chuan's preprocessing.
2. Filter metadata by the final Beauty item universe using `parent_asin`.
3. Select one product image per item from metadata.
4. Add image status fields such as `has_image_url` and `image_embedding_ok`.
5. Encode images with MobileCLIP2-S0 on Apple MPS.
6. Save resumable parquet chunks and a reusable parent-ASIN embedding table.
7. Save a tensor indexed by Yun Chuan's integer `iid`, plus mapping files for CBPR alignment.

Run the notebook first in `RUN_MODE = "smoke"`. After the summary looks good, switch to `RUN_MODE = "full"`.

## One-Time Dependency Setup

Run these commands once in your Jupyter environment if imports fail. They need internet access.

```python
%pip install -U pandas pyarrow pillow requests tqdm huggingface_hub open_clip_torch timm torch torchvision
%pip install -U git+https://github.com/apple/ml-mobileclip.git
```

Notes:

- The first run also downloads the `apple/MobileCLIP2-S0` checkpoint from Hugging Face.
- The official MobileCLIP2 example uses `open_clip.create_model_and_transforms(...)`, `model.eval()`, and `reparameterize_model(...)` before inference.
- This notebook uses `mps` if available and can be configured to stop if MPS is not available.

In [ ]:
# =========================
# Configuration
# =========================
from pathlib import Path

PROJECT_DIR = Path('/Users/frankwang1224/Projects/rcd_sys_proj02')
REVIEW_JSONL = PROJECT_DIR / 'dataset' / 'Beauty_and_Personal_Care.jsonl'
META_JSONL = PROJECT_DIR / 'dataset' / 'meta_Beauty_and_Personal_Care.jsonl'

OUTPUT_DIR = PROJECT_DIR / 'embeddings' / 'image_mobileclip2_s0_beauty'
CHUNK_DIR = OUTPUT_DIR / 'chunks'
RECBole_DATA_DIR = OUTPUT_DIR / 'recbole_data'
ATOMIC_DATASET_DIR = RECBole_DATA_DIR / 'beauty'

# Run smoke first. Change to "full" when the smoke test passes.
RUN_MODE = 'smoke'  # 'smoke' or 'full'
SMOKE_MAX_ITEMS = 200

# Yun Chuan-style filtering parameters.
START_DATE = '2021-01-01'
END_DATE = '2022-12-31'
MIN_RATING = None
USER_MIN_REVIEWS = 5
WARM_USER_MIN_REVIEWS = 10
WARM_ITEM_MIN_REVIEWS = 5
TRAIN_END_CUTOFF_DATE = '2022-08-01'
VALID_END_CUTOFF_DATE = '2022-10-01'
RANDOM_SEED = 42

# Image embedding settings.
MODEL_NAME = 'MobileCLIP2-S0'
HF_REPO_ID = 'apple/MobileCLIP2-S0'
OPENCLIP_PRETRAINED_ALIAS = 'dfndr2b'
TRY_OPENCLIP_ALIAS_FIRST = True
MOBILECLIP_CHECKPOINT_PATH = None  # Optional local .pt checkpoint path.
REQUIRE_MPS = True
ENCODE_BATCH_SIZE = 32
DOWNLOAD_WORKERS = 12
DOWNLOAD_TIMEOUT_SECONDS = 20
PARQUET_COMPRESSION = 'snappy'
CHUNK_SIZE = 500
USER_AGENT = 'Mozilla/5.0 (Macintosh; Intel Mac OS X) AppleWebKit/537.36 BeautyImageEmbedding/1.0'

# Outputs.
FILTERED_REVIEWS_PATH = OUTPUT_DIR / 'beauty_filtered_reviews_yc_logic.csv'
ITEM_UNIVERSE_PATH = OUTPUT_DIR / 'beauty_final_item_universe.csv'
IMAGE_CANDIDATES_PATH = OUTPUT_DIR / 'image_candidates.csv'
IMAGE_MANIFEST_PATH = OUTPUT_DIR / 'image_manifest.csv'
COMBINED_EMBEDDINGS_PATH = OUTPUT_DIR / 'image_embeddings_by_parent_asin.parquet'
ITEM_MAP_PATH = OUTPUT_DIR / 'beauty_item_id_parent_asin_map.csv'
# Debug/intermediate tensor indexed by Yun Chuan-style iid.
# Do NOT use this directly inside a RecBole model.
TENSOR_BY_YC_IID_PATH = OUTPUT_DIR / 'beauty_mobileclip2_s0_image_embeddings_by_yc_iid_debug.pt'
HAS_IMAGE_BY_YC_IID_PATH = OUTPUT_DIR / 'beauty_mobileclip2_s0_has_image_by_yc_iid_debug.pt'
SUMMARY_PATH = OUTPUT_DIR / 'summary.json'

# RecBole-aligned tensor. This is the only safe tensor for actual CBPR training.
# It must be built with RecBole's internal item-token mapping.
BUILD_RECBole_ALIGNED_TENSOR = True
RECBole_ALIGNED_TENSOR_PATH = OUTPUT_DIR / 'beauty_mobileclip2_s0_image_embeddings_recbole_aligned.pt'
HAS_IMAGE_RECBole_ALIGNED_PATH = OUTPUT_DIR / 'beauty_mobileclip2_s0_has_image_recbole_aligned.pt'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHUNK_DIR.mkdir(parents=True, exist_ok=True)
ATOMIC_DATASET_DIR.mkdir(parents=True, exist_ok=True)

print('Project dir:', PROJECT_DIR)
print('Review file:', REVIEW_JSONL)
print('Metadata file:', META_JSONL)
print('Output dir:', OUTPUT_DIR)
print('Run mode:', RUN_MODE)

In [ ]:
# =========================
# Dependency and device checks
# =========================
import importlib.util

required_packages = {
    'pandas': 'pandas',
    'pyarrow': 'pyarrow',
    'PIL': 'pillow',
    'requests': 'requests',
    'tqdm': 'tqdm',
    'torch': 'torch',
    'huggingface_hub': 'huggingface_hub',
    'open_clip': 'open_clip_torch',
    'mobileclip': 'git+https://github.com/apple/ml-mobileclip.git',
}

missing = [pip_name for module_name, pip_name in required_packages.items() if importlib.util.find_spec(module_name) is None]
if missing:
    raise ImportError(
        'Missing packages: ' + ', '.join(sorted(set(missing))) + '\n'
        'Run the dependency setup cell near the top of this notebook.'
    )

import concurrent.futures as futures
import io
import json
import math
import os
import time
from datetime import datetime, timezone
from typing import Any

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import requests
import torch
from PIL import Image, UnidentifiedImageError
from tqdm.auto import tqdm

try:
    from IPython.display import display
except Exception:
    def display(obj):
        print(obj)

if torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
elif REQUIRE_MPS:
    raise RuntimeError('MPS is not available. Check your PyTorch install and macOS Apple Silicon environment.')
else:
    DEVICE = torch.device('cpu')

print('Torch version:', torch.__version__)
print('MPS available:', torch.backends.mps.is_available())
print('Using device:', DEVICE)

In [ ]:
# =========================
# General utilities
# =========================
def stream_jsonl(path: Path):
    with path.open('r', encoding='utf-8') as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                yield json.loads(line)
            except json.JSONDecodeError as exc:
                print(f'Bad JSON at {path}:{line_no}: {exc}')


def date_to_ms(date_str: str | None) -> int | None:
    if date_str is None:
        return None
    return int(pd.Timestamp(date_str, tz='UTC').timestamp() * 1000)


def clean_token_text(value: Any) -> str:
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return ''
    text = str(value)
    return text.replace('\t', ' ').replace('\n', ' ').replace('\r', ' ').replace('"', '').strip()


def safe_float_or_blank(value: Any):
    try:
        if value is None or value == '':
            return ''
        if isinstance(value, float) and math.isnan(value):
            return ''
        return float(value)
    except Exception:
        return ''


def write_json(path: Path, obj: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8') as f:
        json.dump(obj, f, indent=2, sort_keys=True)


def read_done_parent_asins(chunk_dir: Path) -> set[str]:
    done: set[str] = set()
    for path in sorted(chunk_dir.glob('image_embeddings_*.parquet')):
        try:
            table = pq.read_table(path, columns=['parent_asin'])
            done.update(str(x.as_py()) for x in table['parent_asin'])
        except Exception as exc:
            print(f'Warning: could not read done ids from {path}: {exc}')
    return done

In [ ]:
# =========================
# Step 1: Build Beauty interactions using Yun Chuan-style filtering
# =========================
def build_filtered_interactions() -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, dict[str, int], dict[str, int], set[int], set[int]]:
    start_ts = date_to_ms(START_DATE)
    end_ts = date_to_ms(END_DATE)
    rows: list[dict[str, Any]] = []

    for obj in tqdm(stream_jsonl(REVIEW_JSONL), desc='Streaming Beauty reviews'):
        uid = obj.get('user_id')
        parent_asin = obj.get('parent_asin')
        rating = obj.get('rating')
        ts = obj.get('timestamp')

        if uid is None or parent_asin is None or ts is None:
            continue
        if start_ts is not None and ts < start_ts:
            continue
        if end_ts is not None and ts > end_ts:
            continue
        if MIN_RATING is not None and rating is not None and rating < MIN_RATING:
            continue

        rows.append({
            'user_id': str(uid),
            'parent_asin': str(parent_asin),
            'rating': float(rating) if rating is not None else np.nan,
            'timestamp': int(ts),
        })

    df = pd.DataFrame(rows)
    print(f'Loaded after date/rating filter: {len(df):,} rows')
    if df.empty:
        raise RuntimeError('No reviews remained after filtering. Check paths and dates.')

    before = len(df)
    df = df.sort_values('timestamp').drop_duplicates(['user_id', 'parent_asin'], keep='last').reset_index(drop=True)
    print(f'Removed duplicate (user_id, parent_asin) rows: {before - len(df):,}')

    # Match Yun Chuan: create maps before user-min filtering.
    user_ids = set(df['user_id'])
    item_ids = set(df['parent_asin'])
    user_map = {uid: i + 1 for i, uid in enumerate(sorted(user_ids))}
    item_map = {pid: i + 1 for i, pid in enumerate(sorted(item_ids))}
    df['uid'] = df['user_id'].map(user_map).astype(int)
    df['iid'] = df['parent_asin'].map(item_map).astype(int)

    before_users = df['uid'].nunique()
    user_counts = df.groupby('uid').size()
    valid_users = set(user_counts[user_counts >= USER_MIN_REVIEWS].index)
    df = df[df['uid'].isin(valid_users)].copy().reset_index(drop=True)
    print(f'Removed users with < {USER_MIN_REVIEWS} reviews: {before_users - df["uid"].nunique():,}')
    print(f'Remaining reviews: {len(df):,}; users: {df["uid"].nunique():,}; items: {df["iid"].nunique():,}')

    train_end_ts = date_to_ms(TRAIN_END_CUTOFF_DATE)
    valid_end_ts = date_to_ms(VALID_END_CUTOFF_DATE)

    df_train = df[df['timestamp'] <= train_end_ts].copy()
    df_valid = df[(df['timestamp'] > train_end_ts) & (df['timestamp'] <= valid_end_ts)].copy()
    df_test = df[df['timestamp'] > valid_end_ts].copy()

    train_users = set(df_train['uid'].unique())
    df_valid = df_valid[df_valid['uid'].isin(train_users)].copy()
    df_test = df_test[df_test['uid'].isin(train_users)].copy()
    df_all = pd.concat([df_train, df_valid, df_test], ignore_index=True)

    train_user_counts = df_train.groupby('uid').size()
    warm_user_ids = set(train_user_counts[train_user_counts >= WARM_USER_MIN_REVIEWS].index)
    cold_user_ids = set(train_user_counts[train_user_counts < WARM_USER_MIN_REVIEWS].index)

    train_item_counts = df_train.groupby('iid').size()
    warm_item_ids = set(train_item_counts[train_item_counts >= WARM_ITEM_MIN_REVIEWS].index)
    cold_item_ids = set(train_item_counts[train_item_counts < WARM_ITEM_MIN_REVIEWS].index)

    split_summary = pd.DataFrame({
        'split': ['train', 'valid', 'test', 'all'],
        'reviews': [len(df_train), len(df_valid), len(df_test), len(df_all)],
        'users': [df_train['uid'].nunique(), df_valid['uid'].nunique(), df_test['uid'].nunique(), df_all['uid'].nunique()],
        'items': [df_train['iid'].nunique(), df_valid['iid'].nunique(), df_test['iid'].nunique(), df_all['iid'].nunique()],
    })
    display(split_summary)

    df.to_csv(FILTERED_REVIEWS_PATH, index=False)
    item_universe = df_all[['parent_asin', 'iid']].drop_duplicates().sort_values('iid').reset_index(drop=True)
    item_universe.to_csv(ITEM_UNIVERSE_PATH, index=False)

    return df, df_train, df_valid, df_test, df_all, user_map, item_map, warm_user_ids, warm_item_ids


df_reviews_filtered, df_train, df_valid, df_test, df_all, user_map, item_map, warm_user_ids, warm_item_ids = build_filtered_interactions()
final_parent_asins = set(df_all['parent_asin'].unique())
print(f'Final parent_asin universe: {len(final_parent_asins):,}')

In [ ]:
# =========================
# Step 2: Filter metadata and choose one product image per item
# =========================
def image_url_candidates(images: Any) -> list[dict[str, Any]]:
    if not isinstance(images, list):
        return []

    candidates: list[dict[str, Any]] = []
    source_priority = {'hi_res': 0, 'large': 1, 'thumb': 2}
    for image_idx, image_obj in enumerate(images):
        if not isinstance(image_obj, dict):
            continue
        variant = str(image_obj.get('variant') or '').strip()
        is_main = variant.upper() == 'MAIN'
        for source_field in ['hi_res', 'large', 'thumb']:
            url = image_obj.get(source_field)
            if isinstance(url, str) and url.strip().startswith(('http://', 'https://')):
                candidates.append({
                    'selected_image_url': url.strip(),
                    'selected_image_variant': variant,
                    'selected_image_source_field': source_field,
                    'priority': (0 if is_main else 1, source_priority[source_field], image_idx),
                })
    return sorted(candidates, key=lambda x: x['priority'])


def best_metadata_record(existing: dict[str, Any] | None, new: dict[str, Any]) -> dict[str, Any]:
    if existing is None:
        return new
    return new if new['image_priority'] < existing['image_priority'] else existing


def build_image_candidates() -> pd.DataFrame:
    needed = set(final_parent_asins)
    found: dict[str, dict[str, Any]] = {}

    for obj in tqdm(stream_jsonl(META_JSONL), desc='Streaming Beauty metadata'):
        parent_asin = obj.get('parent_asin')
        if parent_asin is None:
            continue
        parent_asin = str(parent_asin)
        if parent_asin not in needed:
            continue

        candidates = image_url_candidates(obj.get('images'))
        if candidates:
            chosen = candidates[0]
            has_image_url = 1
            fail_reason = ''
            image_priority = chosen['priority']
        else:
            chosen = {
                'selected_image_url': '',
                'selected_image_variant': '',
                'selected_image_source_field': '',
            }
            has_image_url = 0
            fail_reason = 'missing_image_url'
            image_priority = (9, 9, 9)

        record = {
            'parent_asin': parent_asin,
            'item_id': int(item_map[parent_asin]),
            'title': clean_token_text(obj.get('title')),
            'store': clean_token_text(obj.get('store')),
            'price': safe_float_or_blank(obj.get('price')),
            'categories': ' > '.join(str(x) for x in obj.get('categories') or []),
            'selected_image_url': chosen['selected_image_url'],
            'selected_image_variant': chosen['selected_image_variant'],
            'selected_image_source_field': chosen['selected_image_source_field'],
            'has_image_url': has_image_url,
            'image_fail_reason': fail_reason,
            'image_priority': image_priority,
        }
        found[parent_asin] = best_metadata_record(found.get(parent_asin), record)

    rows: list[dict[str, Any]] = []
    for parent_asin in sorted(needed, key=lambda pid: item_map[pid]):
        record = found.get(parent_asin)
        if record is None:
            record = {
                'parent_asin': parent_asin,
                'item_id': int(item_map[parent_asin]),
                'title': '',
                'store': '',
                'price': '',
                'categories': '',
                'selected_image_url': '',
                'selected_image_variant': '',
                'selected_image_source_field': '',
                'has_image_url': 0,
                'image_fail_reason': 'metadata_missing',
                'image_priority': (9, 9, 9),
            }
        rows.append(record)

    df = pd.DataFrame(rows).drop(columns=['image_priority'])
    df['image_embedding_ok'] = 0
    df['has_image'] = 0
    df.to_csv(IMAGE_CANDIDATES_PATH, index=False)
    print(f'Image candidate rows: {len(df):,}')
    print(f'Rows with image URL: {int(df["has_image_url"].sum()):,}')
    return df


df_image_candidates = build_image_candidates()
display(df_image_candidates.head())

In [ ]:
# =========================
# Step 3: Write RecBole-style atomic files for this Beauty split
# =========================
def get_user_cold(uid: int) -> int:
    return 0 if uid in warm_user_ids else 1


def get_item_cold(iid: int) -> int:
    return 0 if iid in warm_item_ids else 1


def write_inter_file(path: Path, df: pd.DataFrame) -> None:
    out = df[['uid', 'iid', 'rating', 'timestamp']].copy()
    out.columns = ['user_id:token', 'item_id:token', 'rating:float', 'timestamp:float']
    out.to_csv(path, sep='\t', index=False)
    print(f'Wrote {path} ({len(out):,} rows)')


def write_user_file(path: Path, df_all: pd.DataFrame) -> None:
    uids = sorted(set(int(x) for x in df_all['uid'].unique()))
    out = pd.DataFrame({
        'user_id:token': uids,
        'cold:token': [get_user_cold(uid) for uid in uids],
    })
    out.to_csv(path, sep='\t', index=False)
    print(f'Wrote {path} ({len(out):,} rows)')


def write_item_file(path: Path, df_candidates: pd.DataFrame) -> None:
    out = df_candidates[['item_id', 'parent_asin', 'title', 'store', 'price', 'has_image_url']].copy()
    out['title'] = out['title'].map(clean_token_text)
    out['store'] = out['store'].map(clean_token_text)
    out['price'] = out['price'].map(safe_float_or_blank)
    out['cold'] = out['item_id'].map(lambda iid: get_item_cold(int(iid)))
    out.columns = [
        'item_id:token',
        'parent_asin:token',
        'title:token',
        'store:token',
        'price:float',
        'has_image_url:token',
        'cold:token',
    ]
    out.to_csv(path, sep='\t', index=False)
    print(f'Wrote {path} ({len(out):,} rows)')


prefix = ATOMIC_DATASET_DIR / 'beauty'
write_inter_file(prefix.with_suffix('.train.inter'), df_train)
write_inter_file(prefix.with_suffix('.valid.inter'), df_valid)
write_inter_file(prefix.with_suffix('.test.inter'), df_test)
write_user_file(prefix.with_suffix('.user'), df_all)
write_item_file(prefix.with_suffix('.item'), df_image_candidates)

item_map_df = df_image_candidates[['item_id', 'parent_asin', 'title', 'selected_image_url', 'has_image_url']].copy()
item_map_df['item_cold'] = item_map_df['item_id'].map(lambda iid: get_item_cold(int(iid)))
item_map_df.to_csv(ITEM_MAP_PATH, index=False)
print('Saved item map:', ITEM_MAP_PATH)

In [ ]:
# =========================
# Step 4: Load MobileCLIP2-S0
# =========================
def find_checkpoint_file(snapshot_dir: Path) -> Path:
    candidates: list[Path] = []
    for pattern in ['*.pt', '*.pth', '*.bin', '*.safetensors']:
        candidates.extend(snapshot_dir.rglob(pattern))
    if not candidates:
        raise FileNotFoundError(f'No checkpoint file found under {snapshot_dir}')
    candidates = sorted(candidates, key=lambda p: p.stat().st_size, reverse=True)
    return candidates[0]


def openclip_mobileclip_diagnostics(open_clip_module) -> tuple[list[str], list[tuple[str, str]]]:
    models = sorted([m for m in open_clip_module.list_models() if 'obile' in m.lower()])
    pretrained_raw = open_clip_module.list_pretrained()
    pretrained: list[tuple[str, str]] = []
    for item in pretrained_raw:
        if isinstance(item, (tuple, list)) and len(item) >= 2:
            model_name, tag = str(item[0]), str(item[1])
            if 'obile' in model_name.lower():
                pretrained.append((model_name, tag))
    pretrained = sorted(pretrained)

    print('OpenCLIP Mobile* models available:')
    print(models if models else '  <none>')
    print('OpenCLIP Mobile* pretrained tags available:')
    print(pretrained if pretrained else '  <none>')
    return models, pretrained


def try_openclip_load(open_clip_module, pretrained_arg: str, errors: list[str]):
    try:
        model, _, preprocess = open_clip_module.create_model_and_transforms(
            MODEL_NAME,
            pretrained=pretrained_arg,
        )
        return model, preprocess
    except Exception as exc:
        errors.append(f'open_clip load failed with pretrained={pretrained_arg!r}: {repr(exc)}')
        return None, None


def try_apple_mobileclip_load(checkpoint_path: Path, errors: list[str]):
    try:
        import mobileclip
    except Exception as exc:
        errors.append(f'apple mobileclip import failed: {repr(exc)}')
        return None, None, ''

    if not hasattr(mobileclip, 'create_model_and_transforms'):
        errors.append('apple mobileclip package has no create_model_and_transforms function')
        return None, None, ''

    candidate_arches = [
        MODEL_NAME,
        MODEL_NAME.lower(),
        MODEL_NAME.lower().replace('-', '_'),
    ]
    for arch in candidate_arches:
        try:
            model, _, preprocess = mobileclip.create_model_and_transforms(
                arch,
                pretrained=str(checkpoint_path),
            )
            return model, preprocess, f'apple mobileclip arch={arch}, checkpoint={checkpoint_path}'
        except Exception as exc:
            errors.append(f'apple mobileclip load failed with arch={arch!r}: {repr(exc)}')
    return None, None, ''


def maybe_reparameterize_model(model):
    try:
        from mobileclip.modules.common.mobileone import reparameterize_model
        model = reparameterize_model(model)
        return model, True, ''
    except Exception as exc:
        message = (
            f'Warning: reparameterize_model failed with {type(exc).__name__}: {exc}. '
            'Continuing without reparameterization. If embeddings fail or are unexpectedly slow, '
            'install/update apple/ml-mobileclip and retry.'
        )
        print(message)
        return model, False, message


def load_mobileclip2_s0():
    import open_clip
    from huggingface_hub import snapshot_download

    available_models, available_pretrained = openclip_mobileclip_diagnostics(open_clip)
    available_pretrained_set = set(available_pretrained)
    errors: list[str] = []

    model = None
    preprocess = None
    load_source = ''
    checkpoint_path: Path | None = None

    if MODEL_NAME not in available_models:
        errors.append(
            f'{MODEL_NAME!r} is not registered in this open_clip build. '
            'Install the Apple ml-mobileclip package and/or the patched OpenCLIP build before relying on this notebook.'
        )

    if TRY_OPENCLIP_ALIAS_FIRST and MOBILECLIP_CHECKPOINT_PATH is None:
        if (MODEL_NAME, OPENCLIP_PRETRAINED_ALIAS) in available_pretrained_set:
            model, preprocess = try_openclip_load(open_clip, OPENCLIP_PRETRAINED_ALIAS, errors)
            if model is not None:
                load_source = f'open_clip pretrained alias: {MODEL_NAME}/{OPENCLIP_PRETRAINED_ALIAS}'
        else:
            errors.append(
                f'Skipped alias load because ({MODEL_NAME!r}, {OPENCLIP_PRETRAINED_ALIAS!r}) '
                'was not listed by open_clip.list_pretrained().' 
            )

    if model is None:
        if MOBILECLIP_CHECKPOINT_PATH is not None:
            checkpoint_path = Path(MOBILECLIP_CHECKPOINT_PATH).expanduser()
        else:
            snapshot_dir = Path(snapshot_download(repo_id=HF_REPO_ID))
            checkpoint_path = find_checkpoint_file(snapshot_dir)

        if MODEL_NAME in available_models:
            model, preprocess = try_openclip_load(open_clip, str(checkpoint_path), errors)
            if model is not None:
                load_source = f'open_clip checkpoint path: {checkpoint_path}'

        if model is None:
            model, preprocess, load_source = try_apple_mobileclip_load(checkpoint_path, errors)

    if model is None or preprocess is None:
        diagnostic = '\n'.join(errors)
        raise RuntimeError(
            'Could not load MobileCLIP2-S0. Diagnostics:\n'
            f'{diagnostic}\n\n'
            'Useful checks to run in this kernel:\n'
            "import open_clip\n"
            "print([m for m in open_clip.list_models() if 'obile' in m.lower()])\n"
            "print([p for p in open_clip.list_pretrained() if 'obile' in p[0].lower()])\n"
        )

    model.eval()
    model, reparameterized, reparameterize_warning = maybe_reparameterize_model(model)
    model.eval()
    model.to(DEVICE)

    print('Loaded MobileCLIP2-S0 from', load_source)
    print('Reparameterized:', reparameterized)
    return model, preprocess, load_source, reparameterized, reparameterize_warning


model, preprocess, model_load_source, model_reparameterized, model_reparameterize_warning = load_mobileclip2_s0()

In [ ]:
# =========================
# Step 5: Download and encode helpers
# =========================
def infer_embedding_dim(model, preprocess) -> int:
    dummy = Image.new('RGB', (224, 224), color=(255, 255, 255))
    tensor = preprocess(dummy).unsqueeze(0).to(DEVICE)
    with torch.inference_mode():
        feat = model.encode_image(tensor)
        feat = feat / feat.norm(dim=-1, keepdim=True).clamp_min(1e-12)
    return int(feat.shape[-1])


EMBEDDING_DIM = infer_embedding_dim(model, preprocess)
ZERO_VECTOR = np.zeros(EMBEDDING_DIM, dtype=np.float32)
print('Embedding dim:', EMBEDDING_DIM)


def download_image(row: dict[str, Any]) -> dict[str, Any]:
    url = row.get('selected_image_url') or ''
    result = dict(row)
    result['pil_image'] = None

    if not int(row.get('has_image_url', 0)) or not url:
        result['image_embedding_ok'] = 0
        result['has_image'] = 0
        result['image_fail_reason'] = row.get('image_fail_reason') or 'missing_image_url'
        return result

    try:
        resp = requests.get(
            url,
            timeout=DOWNLOAD_TIMEOUT_SECONDS,
            headers={'User-Agent': USER_AGENT},
        )
        resp.raise_for_status()
        image = Image.open(io.BytesIO(resp.content)).convert('RGB')
        result['pil_image'] = image
        result['image_fail_reason'] = ''
        return result
    except requests.Timeout:
        result['image_fail_reason'] = 'download_timeout'
    except requests.HTTPError as exc:
        result['image_fail_reason'] = f'http_error:{getattr(exc.response, "status_code", "unknown")}'
    except UnidentifiedImageError:
        result['image_fail_reason'] = 'invalid_image_bytes'
    except Exception as exc:
        result['image_fail_reason'] = f'download_or_open_error:{type(exc).__name__}'

    result['image_embedding_ok'] = 0
    result['has_image'] = 0
    return result


def encode_images(downloaded_rows: list[dict[str, Any]]) -> list[dict[str, Any]]:
    ok_rows = [r for r in downloaded_rows if r.get('pil_image') is not None]
    by_parent: dict[str, np.ndarray] = {}

    for start in range(0, len(ok_rows), ENCODE_BATCH_SIZE):
        batch = ok_rows[start:start + ENCODE_BATCH_SIZE]
        try:
            tensors = torch.stack([preprocess(r['pil_image']) for r in batch]).to(DEVICE)
            with torch.inference_mode():
                feats = model.encode_image(tensors)
                feats = feats / feats.norm(dim=-1, keepdim=True).clamp_min(1e-12)
            feats_np = feats.detach().cpu().float().numpy().astype(np.float32)
            for row, feat in zip(batch, feats_np):
                by_parent[str(row['parent_asin'])] = feat
        except Exception as batch_exc:
            # Fall back to one-by-one so one bad image does not poison the whole batch.
            for row in batch:
                try:
                    tensor = preprocess(row['pil_image']).unsqueeze(0).to(DEVICE)
                    with torch.inference_mode():
                        feat = model.encode_image(tensor)
                        feat = feat / feat.norm(dim=-1, keepdim=True).clamp_min(1e-12)
                    by_parent[str(row['parent_asin'])] = feat.detach().cpu().float().numpy()[0].astype(np.float32)
                except Exception:
                    row['image_fail_reason'] = f'encoder_error:{type(batch_exc).__name__}'

    encoded: list[dict[str, Any]] = []
    for row in downloaded_rows:
        parent_asin = str(row['parent_asin'])
        row = dict(row)
        row.pop('pil_image', None)
        feat = by_parent.get(parent_asin)
        if feat is None:
            row['image_embedding'] = ZERO_VECTOR
            row['image_embedding_ok'] = 0
            row['has_image'] = 0
            if not row.get('image_fail_reason'):
                row['image_fail_reason'] = 'encoder_error'
        else:
            row['image_embedding'] = feat.astype(np.float32)
            row['image_embedding_ok'] = 1
            row['has_image'] = 1
            row['image_fail_reason'] = ''
        row['embedding_model'] = HF_REPO_ID
        row['embedding_dim'] = EMBEDDING_DIM
        row['embedding_dtype'] = 'float32'
        encoded.append(row)
    return encoded

In [ ]:
# =========================
# Step 6: Chunk writer and embedding run
# =========================
def write_embedding_chunk(records: list[dict[str, Any]], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = path.with_name(path.name + '.tmp')

    table = pa.table({
        'parent_asin': pa.array([str(r['parent_asin']) for r in records], type=pa.string()),
        'item_id': pa.array([int(r['item_id']) for r in records], type=pa.int64()),
        'title': pa.array([clean_token_text(r.get('title')) for r in records], type=pa.string()),
        'selected_image_url': pa.array([str(r.get('selected_image_url') or '') for r in records], type=pa.string()),
        'selected_image_variant': pa.array([str(r.get('selected_image_variant') or '') for r in records], type=pa.string()),
        'selected_image_source_field': pa.array([str(r.get('selected_image_source_field') or '') for r in records], type=pa.string()),
        'has_image_url': pa.array([int(r.get('has_image_url', 0)) for r in records], type=pa.int8()),
        'image_embedding_ok': pa.array([int(r.get('image_embedding_ok', 0)) for r in records], type=pa.int8()),
        'has_image': pa.array([int(r.get('has_image', 0)) for r in records], type=pa.int8()),
        'image_fail_reason': pa.array([str(r.get('image_fail_reason') or '') for r in records], type=pa.string()),
        'embedding_model': pa.array([str(r.get('embedding_model') or HF_REPO_ID) for r in records], type=pa.string()),
        'embedding_dim': pa.array([int(r.get('embedding_dim') or EMBEDDING_DIM) for r in records], type=pa.int32()),
        'embedding_dtype': pa.array([str(r.get('embedding_dtype') or 'float32') for r in records], type=pa.string()),
        'image_embedding': pa.array([np.asarray(r['image_embedding'], dtype=np.float32).tolist() for r in records], type=pa.list_(pa.float32())),
    })
    pq.write_table(table, tmp_path, compression=PARQUET_COMPRESSION)
    tmp_path.replace(path)


def process_embeddings() -> None:
    df = pd.read_csv(IMAGE_CANDIDATES_PATH)
    done = read_done_parent_asins(CHUNK_DIR)
    pending = df[~df['parent_asin'].astype(str).isin(done)].copy()

    if RUN_MODE == 'smoke':
        pending = pending.head(SMOKE_MAX_ITEMS).copy()
    elif RUN_MODE != 'full':
        raise ValueError("RUN_MODE must be 'smoke' or 'full'")

    print(f'Already completed items: {len(done):,}')
    print(f'Pending items for this run: {len(pending):,}')
    if pending.empty:
        return

    existing_chunk_indices = []
    for path in CHUNK_DIR.glob('image_embeddings_*.parquet'):
        try:
            existing_chunk_indices.append(int(path.stem.split('_')[-1]))
        except Exception:
            pass
    next_chunk_idx = max(existing_chunk_indices, default=-1) + 1

    rows = pending.to_dict('records')
    for start in range(0, len(rows), CHUNK_SIZE):
        chunk_rows = rows[start:start + CHUNK_SIZE]
        chunk_path = CHUNK_DIR / f'image_embeddings_{next_chunk_idx:05d}.parquet'
        next_chunk_idx += 1

        with futures.ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS) as executor:
            downloaded = list(tqdm(
                executor.map(download_image, chunk_rows),
                total=len(chunk_rows),
                desc=f'Downloading {chunk_path.name}',
            ))

        encoded = encode_images(downloaded)
        write_embedding_chunk(encoded, chunk_path)
        ok_count = sum(int(r['image_embedding_ok']) for r in encoded)
        print(f'Wrote {chunk_path} ({len(encoded):,} rows, {ok_count:,} embedded ok)')


process_embeddings()

In [ ]:
# =========================
# Step 7: Combine chunks, write manifest, summary, and debug yc-iid tensor
# =========================
def combine_chunks() -> pa.Table:
    chunk_paths = sorted(CHUNK_DIR.glob('image_embeddings_*.parquet'))
    if not chunk_paths:
        raise RuntimeError('No embedding chunks found. Run the embedding cell first.')
    tables = [pq.read_table(path) for path in chunk_paths]
    table = pa.concat_tables(tables, promote_options='default')
    pq.write_table(table, COMBINED_EMBEDDINGS_PATH, compression=PARQUET_COMPRESSION)
    print('Wrote combined embeddings:', COMBINED_EMBEDDINGS_PATH)
    return table


def write_manifest_from_table() -> pd.DataFrame:
    columns = [
        'parent_asin', 'item_id', 'title', 'selected_image_url',
        'selected_image_variant', 'selected_image_source_field',
        'has_image_url', 'image_embedding_ok', 'has_image', 'image_fail_reason',
        'embedding_model', 'embedding_dim', 'embedding_dtype',
    ]
    table = pq.read_table(COMBINED_EMBEDDINGS_PATH, columns=columns)
    df_manifest = table.to_pandas()
    df_manifest.sort_values('item_id').to_csv(IMAGE_MANIFEST_PATH, index=False)
    print('Wrote manifest:', IMAGE_MANIFEST_PATH)
    return df_manifest


def build_yc_iid_tensor() -> tuple[torch.Tensor, torch.Tensor]:
    table = pq.read_table(COMBINED_EMBEDDINGS_PATH, columns=['parent_asin', 'item_id', 'image_embedding', 'has_image'])
    df = table.to_pandas()
    max_iid = int(df['item_id'].max())
    tensor = torch.zeros((max_iid + 1, EMBEDDING_DIM), dtype=torch.float32)
    has_image = torch.zeros((max_iid + 1,), dtype=torch.bool)

    for row in tqdm(df.itertuples(index=False), total=len(df), desc='Building iid tensor'):
        iid = int(row.item_id)
        vec = np.asarray(row.image_embedding, dtype=np.float32)
        if vec.shape[0] != EMBEDDING_DIM:
            raise ValueError(f'Bad embedding dim for item_id={iid}: {vec.shape[0]} != {EMBEDDING_DIM}')
        tensor[iid] = torch.from_numpy(vec)
        has_image[iid] = bool(int(row.has_image))

    tensor[0].zero_()
    has_image[0] = False
    torch.save(tensor, TENSOR_BY_YC_IID_PATH)
    torch.save(has_image, HAS_IMAGE_BY_YC_IID_PATH)
    print('Wrote DEBUG yc-iid tensor:', TENSOR_BY_YC_IID_PATH, tuple(tensor.shape))
    print('Wrote DEBUG yc-iid has-image flags:', HAS_IMAGE_BY_YC_IID_PATH, tuple(has_image.shape))
    print('Important: this yc-iid tensor is NOT safe for direct RecBole CBPR training.')
    return tensor, has_image


def write_summary(df_manifest: pd.DataFrame, tensor: torch.Tensor, has_image: torch.Tensor) -> dict[str, Any]:
    summary = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'run_mode': RUN_MODE,
        'review_jsonl': str(REVIEW_JSONL),
        'meta_jsonl': str(META_JSONL),
        'model_name': MODEL_NAME,
        'hf_repo_id': HF_REPO_ID,
        'model_load_source': model_load_source,
        'model_reparameterized': bool(model_reparameterized),
        'model_reparameterize_warning': model_reparameterize_warning,
        'device': str(DEVICE),
        'embedding_dim': EMBEDDING_DIM,
        'embedding_dtype': 'float32',
        'final_parent_asins': int(len(final_parent_asins)),
        'chunk_rows_combined': int(len(df_manifest)),
        'items_with_image_url': int(df_manifest['has_image_url'].sum()),
        'items_successfully_embedded': int(df_manifest['image_embedding_ok'].sum()),
        'items_missing_or_failed_image': int((df_manifest['image_embedding_ok'] == 0).sum()),
        'image_embedding_ok_rate': float(df_manifest['image_embedding_ok'].mean()) if len(df_manifest) else 0.0,
        'tensor_shape_by_yc_iid': list(tensor.shape),
        'tensor_has_image_count': int(has_image.sum().item()),
        'outputs': {
            'image_manifest_csv': str(IMAGE_MANIFEST_PATH),
            'combined_parent_asin_parquet': str(COMBINED_EMBEDDINGS_PATH),
            'item_map_csv': str(ITEM_MAP_PATH),
            'debug_tensor_by_yc_iid_pt': str(TENSOR_BY_YC_IID_PATH),
            'debug_has_image_by_yc_iid_pt': str(HAS_IMAGE_BY_YC_IID_PATH),
            'recbole_aligned_tensor_pt': str(RECBole_ALIGNED_TENSOR_PATH),
            'recbole_aligned_has_image_pt': str(HAS_IMAGE_RECBole_ALIGNED_PATH),
        },
    }
    write_json(SUMMARY_PATH, summary)
    print('Wrote summary:', SUMMARY_PATH)
    return summary


combined_table = combine_chunks()
df_manifest = write_manifest_from_table()
tensor_by_iid, has_image_by_iid = build_yc_iid_tensor()
summary = write_summary(df_manifest, tensor_by_iid, has_image_by_iid)
summary

In [ ]:
# =========================
# Step 8: QA checks for the debug yc-iid tensor
# =========================
def run_qa_checks() -> None:
    assert TENSOR_BY_YC_IID_PATH.exists(), 'Missing iid tensor'
    assert COMBINED_EMBEDDINGS_PATH.exists(), 'Missing combined parquet'
    assert IMAGE_MANIFEST_PATH.exists(), 'Missing manifest CSV'

    tensor = torch.load(TENSOR_BY_YC_IID_PATH, map_location='cpu')
    has_image = torch.load(HAS_IMAGE_BY_YC_IID_PATH, map_location='cpu')
    manifest = pd.read_csv(IMAGE_MANIFEST_PATH)

    assert tensor.dtype == torch.float32, f'Expected float32 tensor, got {tensor.dtype}'
    assert tensor.ndim == 2, f'Expected 2D tensor, got shape {tuple(tensor.shape)}'
    assert tensor.shape[1] == EMBEDDING_DIM, f'Expected dim {EMBEDDING_DIM}, got {tensor.shape[1]}'
    assert torch.all(tensor[0] == 0), 'Row 0 should be zero padding'
    assert torch.isfinite(tensor).all(), 'Tensor has NaN or inf'
    assert has_image.shape[0] == tensor.shape[0], 'has_image flag length mismatch'

    ok_manifest_count = int(manifest['image_embedding_ok'].sum())
    ok_tensor_count = int(has_image.sum().item())
    assert ok_manifest_count == ok_tensor_count, f'Manifest ok count {ok_manifest_count} != tensor ok count {ok_tensor_count}'

    zero_failed = tensor[~has_image]
    if len(zero_failed):
        assert torch.all(zero_failed == 0), 'Missing/failed image rows should be zero vectors'

    print('Debug yc-iid tensor QA checks passed.')
    print('Debug yc-iid tensor shape:', tuple(tensor.shape))
    print('Reminder: do not use this debug tensor directly in RecBole CBPR.')
    print('Successful image embeddings:', ok_tensor_count)
    print('Manifest rows:', len(manifest))
    display(manifest.sample(min(10, len(manifest)), random_state=RANDOM_SEED))


run_qa_checks()

## Build the RecBole-Aligned Training Tensor

This is not optional for CBPR.

The debug tensor `beauty_mobileclip2_s0_image_embeddings_by_yc_iid_debug.pt` is indexed by Yun Chuan-style `iid`. RecBole re-tokenizes `item_id:token` into its own internal IDs. Those index spaces are not guaranteed to match.

For actual CBPR training, build and use:

```text
beauty_mobileclip2_s0_image_embeddings_recbole_aligned.pt
```

This cell loads the same RecBole atomic files written above, reads RecBole's internal item-token mapping, and writes the image tensor in RecBole internal item-ID order.

In [ ]:
# =========================
# Step 9: Build RecBole-aligned training tensor
# =========================
def build_recbole_aligned_tensor() -> tuple[torch.Tensor, torch.Tensor, dict[str, Any]]:
    if not BUILD_RECBole_ALIGNED_TENSOR:
        raise RuntimeError(
            'BUILD_RECBole_ALIGNED_TENSOR is False. For CBPR training it must be True. '
            'The yc-iid debug tensor is not safe to index with RecBole internal item IDs.'
        )

    # RecBole 1.2.1 still references NumPy aliases removed in NumPy 2.x.
    # Patch them before constructing Config so the notebook works in modern envs.
    if not hasattr(np, 'float_'):
        np.float_ = np.float64
    if not hasattr(np, 'float'):
        np.float = float
    if not hasattr(np, 'int'):
        np.int = int
    if not hasattr(np, 'complex_'):
        np.complex_ = np.complex128
    if not hasattr(np, 'complex'):
        np.complex = complex
    if not hasattr(np, 'object_'):
        np.object_ = object
    if not hasattr(np, 'unicode_'):
        np.unicode_ = np.str_
    if not hasattr(np, 'unicode'):
        np.unicode = str

    try:
        from recbole.config import Config
        from recbole.data import create_dataset
    except Exception as exc:
        raise ImportError(
            'RecBole is required to build the training-safe aligned tensor. '
            'Run this cell in the same environment used for CBPR training, after installing RecBole. '
            'Do not use the yc-iid debug tensor as a substitute.'
        ) from exc

    config_dict = {
        'data_path': str(RECBole_DATA_DIR),
        'dataset': 'beauty',
        'USER_ID_FIELD': 'user_id',
        'ITEM_ID_FIELD': 'item_id',
        'benchmark_filename': ['train', 'valid', 'test'],
        'load_col': {
            'inter': ['user_id', 'item_id'],
        },
    }
    import warnings

    config = Config(model='BPR', config_dict=config_dict)
    with warnings.catch_warnings():
        if hasattr(pd.errors, 'ChainedAssignmentError'):
            warnings.simplefilter('ignore', pd.errors.ChainedAssignmentError)
        dataset = create_dataset(config)

    parent_table = pq.read_table(COMBINED_EMBEDDINGS_PATH, columns=['parent_asin', 'image_embedding', 'has_image'])
    parent_df = parent_table.to_pandas()
    emb_by_parent = {
        str(row.parent_asin): np.asarray(row.image_embedding, dtype=np.float32)
        for row in parent_df.itertuples(index=False)
    }
    has_image_by_parent = {
        str(row.parent_asin): bool(int(row.has_image))
        for row in parent_df.itertuples(index=False)
    }

    yc_map = pd.read_csv(ITEM_MAP_PATH)
    iid_to_parent = {str(int(row.item_id)): str(row.parent_asin) for row in yc_map.itertuples(index=False)}

    id_tokens = dataset.field2id_token[dataset.iid_field]
    recbole_tensor = torch.zeros((len(id_tokens), EMBEDDING_DIM), dtype=torch.float32)
    recbole_has_image = torch.zeros((len(id_tokens),), dtype=torch.bool)

    missing_parent_for_token = 0
    missing_embedding_for_parent = 0

    for internal_id, token in enumerate(id_tokens):
        token = str(token)
        if internal_id == 0 or token in ('[PAD]', 'PAD'):
            continue

        parent_asin = iid_to_parent.get(token)
        if parent_asin is None:
            missing_parent_for_token += 1
            continue

        vec = emb_by_parent.get(parent_asin)
        if vec is None:
            missing_embedding_for_parent += 1
            continue

        if vec.shape[0] != EMBEDDING_DIM:
            raise ValueError(f'Bad embedding dim for parent_asin={parent_asin}: {vec.shape[0]} != {EMBEDDING_DIM}')

        recbole_tensor[internal_id] = torch.from_numpy(vec.copy())
        recbole_has_image[internal_id] = has_image_by_parent.get(parent_asin, False)

    recbole_tensor[0].zero_()
    recbole_has_image[0] = False

    assert recbole_tensor.dtype == torch.float32
    assert recbole_tensor.ndim == 2
    assert recbole_tensor.shape[1] == EMBEDDING_DIM
    assert torch.all(recbole_tensor[0] == 0)
    assert torch.isfinite(recbole_tensor).all()
    failed_rows = recbole_tensor[~recbole_has_image]
    if len(failed_rows):
        assert torch.all(failed_rows == 0), 'Rows without successful image embeddings should be all-zero vectors.'

    torch.save(recbole_tensor, RECBole_ALIGNED_TENSOR_PATH)
    torch.save(recbole_has_image, HAS_IMAGE_RECBole_ALIGNED_PATH)

    alignment_summary = {
        'recbole_num_item_tokens': int(len(id_tokens)),
        'recbole_aligned_tensor_shape': list(recbole_tensor.shape),
        'recbole_aligned_has_image_count': int(recbole_has_image.sum().item()),
        'missing_parent_for_recbole_token': int(missing_parent_for_token),
        'missing_embedding_for_parent': int(missing_embedding_for_parent),
        'recbole_aligned_tensor_pt': str(RECBole_ALIGNED_TENSOR_PATH),
        'recbole_aligned_has_image_pt': str(HAS_IMAGE_RECBole_ALIGNED_PATH),
    }

    if SUMMARY_PATH.exists():
        with SUMMARY_PATH.open('r', encoding='utf-8') as f:
            summary = json.load(f)
        summary.update(alignment_summary)
        summary.setdefault('outputs', {})['recbole_aligned_tensor_pt'] = str(RECBole_ALIGNED_TENSOR_PATH)
        summary.setdefault('outputs', {})['recbole_aligned_has_image_pt'] = str(HAS_IMAGE_RECBole_ALIGNED_PATH)
        write_json(SUMMARY_PATH, summary)

    print('Wrote RecBole-aligned TRAINING tensor:', RECBole_ALIGNED_TENSOR_PATH, tuple(recbole_tensor.shape))
    print('Wrote RecBole-aligned has-image flags:', HAS_IMAGE_RECBole_ALIGNED_PATH, tuple(recbole_has_image.shape))
    print('This is the tensor to use in CBPR training.')
    print(alignment_summary)
    return recbole_tensor, recbole_has_image, alignment_summary


recbole_tensor, recbole_has_image, alignment_summary = build_recbole_aligned_tensor()

## How to Use in CBPR Later

Use the RecBole-aligned tensor, not the yc-iid debug tensor.

```python
config_dict = {
    # ... existing config ...
    'image_embedding_path': '/Users/frankwang1224/Projects/rcd_sys_proj02/embeddings/image_mobileclip2_s0_beauty/beauty_mobileclip2_s0_image_embeddings_recbole_aligned.pt',
}
```

In the CBPR model:

```python
image_embeddings = torch.load(config['image_embedding_path'], map_location=config['device'])
self.register_buffer('item_image', image_embeddings)
self.user_image = nn.Embedding(self.n_users, 32)
self.image_proj = nn.Linear(image_embeddings.shape[1], 32, bias=False)
```

Then score with RecBole internal item IDs:

```python
i_image = self.image_proj(self.item_image[item])
image_score = torch.mul(u_image, i_image).sum(-1)
```

Full score:

```text
score(u, i)
= user_collab(u) dot item_collab(i)
+ user_image(u) dot image_proj(item_image(i))
```

Do not train MobileCLIP2-S0 during CBPR. Keep the saved image embeddings frozen and let BPR loss train the projection layer and user image-preference vectors.

Critical warning:

```text
Do not load beauty_mobileclip2_s0_image_embeddings_by_yc_iid_debug.pt in CBPR.
It is indexed by Yun Chuan-style iid, not by RecBole internal item ID.
```